In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import (f1_score, balanced_accuracy_score,
                              recall_score, confusion_matrix)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')
import os
os.makedirs('outputs', exist_ok=True)

# Load Nepal's verified clean data
nepal = pd.read_csv('data/processed/nepal_macro_clean.csv',
                     index_col=0, parse_dates=True)

print(f"✅ Nepal data loaded: {nepal.shape}")
print(f"   Columns: {list(nepal.columns)}")
print(f"   Years:   {nepal.index[0].year} → {nepal.index[-1].year}")

✅ Nepal data loaded: (31, 12)
   Columns: ['gdp_growth', 'inflation', 'unemployment', 'remittances_pct_gdp', 'forex_reserves_months', 'gross_investment_gdp', 'consumption_growth', 'exports_pct_gdp', 'imports_pct_gdp', 'india_gdp_growth', 'india_inflation', 'distress']
   Years:   1995 → 2025


In [2]:
# ================================================
# FETCH SOUTH ASIA DATA
# ================================================
# Same 5 raw indicators for 4 additional countries
# We'll engineer features from these later
# ================================================
import wbgapi as wb

def fetch_single(code, country, name, mrv=30):
    raw = wb.data.DataFrame(code, country, mrv=mrv)
    s = raw.T.iloc[:, 0]
    s.name = name
    s.index = s.index.str.replace('YR', '', regex=False)
    s.index = pd.to_datetime(s.index, format='%Y')
    return s.sort_index()

COUNTRIES = {
    'NPL': 'Nepal',
    'IND': 'India',
    'BGD': 'Bangladesh',
    'LKA': 'Sri Lanka',
    'PAK': 'Pakistan',
}

INDICATORS = {
    'NY.GDP.MKTP.KD.ZG' : 'gdp_growth',
    'FP.CPI.TOTL.ZG'    : 'inflation',
    'SL.UEM.TOTL.ZS'    : 'unemployment',
    'BX.TRF.PWKR.DT.GD.ZS': 'remittances_pct_gdp',
    'FI.RES.TOTL.MO'    : 'forex_reserves_months',
}

all_data = {}

for code, country_name in COUNTRIES.items():
    print(f"📥 Fetching {country_name}...")
    frames = []
    for ind_code, ind_name in INDICATORS.items():
        try:
            s = fetch_single(ind_code, code, ind_name, mrv=30)
            frames.append(s)
        except Exception as e:
            print(f"   ⚠️  {ind_name} failed: {e}")

    df_country = pd.concat(frames, axis=1)
    df_country['country'] = country_name
    all_data[country_name] = df_country
    print(f"   ✅ {country_name}: {df_country.shape}")

print("\n✅ All countries fetched!")

📥 Fetching Nepal...
   ✅ Nepal: (31, 6)
📥 Fetching India...
   ✅ India: (31, 6)
📥 Fetching Bangladesh...
   ✅ Bangladesh: (31, 6)
📥 Fetching Sri Lanka...
   ✅ Sri Lanka: (31, 6)
📥 Fetching Pakistan...
   ✅ Pakistan: (31, 6)

✅ All countries fetched!


In [3]:
# ================================================
# DEFINE DISTRESS FOR EACH COUNTRY
# ================================================
# Each country has different thresholds and
# known crisis events. We define distress
# carefully for each one.
# ================================================

DISTRESS_RULES = {
    'Nepal': {
        'gdp_threshold': 2.0,
        'forex_threshold': 6.0,
        'crisis_years': [2015, 2016, 2020, 2021, 2022, 2023],
    },
    'India': {
        'gdp_threshold': 4.0,    # India needs higher growth floor
        'forex_threshold': 8.0,  # Larger economy needs more reserves
        'crisis_years': [2002, 2009, 2020],  # Drought, GFC, COVID
    },
    'Bangladesh': {
        'gdp_threshold': 4.0,
        'forex_threshold': 5.0,
        'crisis_years': [2020, 2022],  # COVID, forex crisis
    },
    'Sri Lanka': {
        'gdp_threshold': 2.0,
        'forex_threshold': 4.0,
        'crisis_years': [2020, 2021, 2022],  # COVID + 2022 bankruptcy
    },
    'Pakistan': {
        'gdp_threshold': 2.0,
        'forex_threshold': 3.0,  # Pakistan frequently has low reserves
        'crisis_years': [2019, 2020, 2023],  # IMF bailouts, COVID
    },
}

for country_name, rules in DISTRESS_RULES.items():
    df = all_data[country_name]
    df['distress'] = 0

    # GDP rule — only from 2000 onwards
    gdp_mask = (df.index.year >= 2000) & \
               df['gdp_growth'].notna() & \
               (df['gdp_growth'] < rules['gdp_threshold'])
    df.loc[gdp_mask, 'distress'] = 1

    # Forex rule — only from 2000 onwards
    forex_mask = (df.index.year >= 2000) & \
                 (df['forex_reserves_months'] < rules['forex_threshold'])
    df.loc[forex_mask, 'distress'] = 1

    # Known crisis years
    for yr in rules['crisis_years']:
        mask = df.index.year == yr
        if mask.any():
            df.loc[mask, 'distress'] = 1

    # 2024/25 = normal for all countries
    df.loc[df.index.year >= 2024, 'distress'] = 0

    distress_yrs = df[df['distress']==1].index.year.tolist()
    rate = df['distress'].mean()
    print(f"{country_name:<12} distress={rate:.1%}  years={distress_yrs}")

Nepal        distress=22.6%  years=[2002, 2015, 2016, 2020, 2021, 2022, 2023]
India        distress=48.4%  years=[2000, 2001, 2002, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2018, 2019, 2020, 2022, 2023]
Bangladesh   distress=51.6%  years=[2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2010, 2011, 2012, 2013, 2020, 2022, 2023]
Sri Lanka    distress=71.0%  years=[2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
Pakistan     distress=29.0%  years=[2000, 2008, 2010, 2013, 2018, 2019, 2020, 2022, 2023]


In [4]:
# Diagnose what's causing high distress rates
for country_name in ['India', 'Bangladesh', 'Sri Lanka']:
    df = all_data[country_name]
    print(f"\n{country_name}:")
    print(f"  GDP growth range: {df['gdp_growth'].min():.1f}% to {df['gdp_growth'].max():.1f}%")
    print(f"  Forex range: {df['forex_reserves_months'].min():.1f} to {df['forex_reserves_months'].max():.1f} months")
    print(f"  Years with GDP < threshold:")
    rules = DISTRESS_RULES[country_name]
    low_gdp = df[df['gdp_growth'] < rules['gdp_threshold']]
    print(f"    {low_gdp.index.year.tolist()}")
    print(f"  Years with forex < threshold:")
    low_forex = df[df['forex_reserves_months'] < rules['forex_threshold']]
    print(f"    {low_forex.index.year.tolist()}")


India:
  GDP growth range: -5.8% to 9.7%
  Forex range: 5.0 to 12.9 months
  Years with GDP < threshold:
    [2000, 2002, 2008, 2019, 2020]
  Years with forex < threshold:
    [1995, 1996, 1997, 1998, 1999, 2000, 2001, 2008, 2010, 2011, 2012, 2013, 2014, 2018, 2022, 2023, 2024]

Bangladesh:
  GDP growth range: 3.4% to 7.9%
  Forex range: 1.6 to 8.7 months
  Years with GDP < threshold:
    [2002, 2020]
  Years with forex < threshold:
    [1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2010, 2011, 2012, 2013, 2022, 2023, 2024]

Sri Lanka:
  GDP growth range: -7.3% to 8.7%
  Forex range: 1.1 to 5.3 months
  Years with GDP < threshold:
    [2001, 2019, 2020, 2022, 2023]
  Years with forex < threshold:
    [1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [5]:
# ================================================
# REVISED DISTRESS RULES — NO FOREX FOR NON-NEPAL
# ================================================

DISTRESS_RULES_V2 = {
    'Nepal': {
        'gdp_threshold': 2.0,
        'forex_threshold': 6.0,   # NRB official threshold
        'use_forex': True,
        'crisis_years': [2015, 2016, 2020, 2021, 2022, 2023],
    },
    'India': {
        'gdp_threshold': 4.0,
        'use_forex': False,        # forex data unreliable for India
        'crisis_years': [2002, 2009, 2020],
    },
    'Bangladesh': {
        'gdp_threshold': 4.0,
        'use_forex': False,
        'crisis_years': [2020, 2022],
    },
    'Sri Lanka': {
        'gdp_threshold': 1.0,     # Sri Lanka had genuine collapses
        'use_forex': False,
        'crisis_years': [2001, 2020, 2021, 2022],  # civil war end, COVID, bankruptcy
    },
    'Pakistan': {
        'gdp_threshold': 2.0,
        'use_forex': False,
        'crisis_years': [2019, 2020, 2023],
    },
}

for country_name, rules in DISTRESS_RULES_V2.items():
    df = all_data[country_name]
    df['distress'] = 0

    # GDP rule — only from 2000 onwards
    gdp_mask = (df.index.year >= 2000) & \
               df['gdp_growth'].notna() & \
               (df['gdp_growth'] < rules['gdp_threshold'])
    df.loc[gdp_mask, 'distress'] = 1

    # Forex rule — Nepal only
    if rules.get('use_forex'):
        forex_mask = (df.index.year >= 2000) & \
                     (df['forex_reserves_months'] < rules['forex_threshold'])
        df.loc[forex_mask, 'distress'] = 1

    # Known crisis years
    for yr in rules['crisis_years']:
        mask = df.index.year == yr
        if mask.any():
            df.loc[mask, 'distress'] = 1

    # 2024/25 = normal for all
    df.loc[df.index.year >= 2024, 'distress'] = 0

    distress_yrs = df[df['distress']==1].index.year.tolist()
    rate = df['distress'].mean()
    status = "✅" if 0.10 <= rate <= 0.40 else "⚠️ "
    print(f"{status} {country_name:<12} distress={rate:.1%}  years={distress_yrs}")

✅ Nepal        distress=22.6%  years=[2002, 2015, 2016, 2020, 2021, 2022, 2023]
✅ India        distress=19.4%  years=[2000, 2002, 2008, 2009, 2019, 2020]
⚠️  Bangladesh   distress=9.7%  years=[2002, 2020, 2022]
✅ Sri Lanka    distress=19.4%  years=[2001, 2019, 2020, 2021, 2022, 2023]
✅ Pakistan     distress=12.9%  years=[2010, 2019, 2020, 2023]


In [6]:
# ================================================
# BUILD SOUTH ASIA PANEL DATASET
# ================================================
# Stack all 5 countries into one DataFrame
# Each row = one country-year observation
# ================================================

panel_frames = []

for country_name, df_country in all_data.items():
    df_c = df_country.copy()
    df_c['country'] = country_name
    panel_frames.append(df_c)

panel_raw = pd.concat(panel_frames, axis=0)
panel_raw = panel_raw.sort_index()

print(f"✅ Panel dataset created!")
print(f"   Total rows:    {len(panel_raw)}")
print(f"   Columns:       {list(panel_raw.columns)}")
print(f"   Countries:     {panel_raw['country'].unique().tolist()}")
print(f"   Distress rate: {panel_raw['distress'].mean():.1%}")
print(f"\nRows per country:")
print(panel_raw.groupby('country')['distress'].agg(['count','sum','mean'])
      .rename(columns={'count':'rows','sum':'distress_yrs','mean':'rate'})
      .round(3).to_string())

✅ Panel dataset created!
   Total rows:    155
   Columns:       ['gdp_growth', 'inflation', 'unemployment', 'remittances_pct_gdp', 'forex_reserves_months', 'country', 'distress']
   Countries:     ['Nepal', 'Bangladesh', 'Sri Lanka', 'Pakistan', 'India']
   Distress rate: 16.8%

Rows per country:
            rows  distress_yrs   rate
country                              
Bangladesh    31             3  0.097
India         31             6  0.194
Nepal         31             7  0.226
Pakistan      31             4  0.129
Sri Lanka     31             6  0.194


In [7]:
# ================================================
# FEATURE ENGINEERING FOR PANEL DATA
# ================================================
# Same 5 features we used for Nepal-only model
# But engineered separately per country
# to avoid mixing country time series
# ================================================

def engineer_features(df):
    """Engineer the same 5 features for one country"""
    d = df.copy()

    # 1. unemployment_roll3_mean
    d['unemployment_roll3_mean'] = (
        d['unemployment'].rolling(window=3, min_periods=2).mean()
    )

    # 2. gdp_growth_roll3_std
    d['gdp_growth_roll3_std'] = (
        d['gdp_growth'].rolling(window=3, min_periods=2).std()
    )

    # 3. consumption_growth_roll3_mean
    # Use GDP growth as proxy — consumption not in panel fetch
    # (correlated with GDP growth across all South Asian economies)
    d['consumption_growth_roll3_mean'] = (
        d['gdp_growth'].rolling(window=3, min_periods=2).mean()
    )

    # 4. forex_rapid_decline
    d['forex_rapid_decline'] = (
        (d['forex_reserves_months'].diff(1) < -3.0)
        .astype(int)
    )

    # 5. remittances_pct_gdp_lag1
    d['remittances_pct_gdp_lag1'] = d['remittances_pct_gdp'].shift(1)

    return d

# Engineer features for each country separately
engineered_frames = []
for country_name, df_country in all_data.items():
    df_eng = engineer_features(df_country)
    engineered_frames.append(df_eng)

panel = pd.concat(engineered_frames, axis=0)
panel = panel.sort_index()

FEATURE_COLS = [
    'unemployment_roll3_mean',
    'gdp_growth_roll3_std',
    'consumption_growth_roll3_mean',
    'forex_rapid_decline',
    'remittances_pct_gdp_lag1',
]

# Drop rows with NaN in features
panel_clean = panel.dropna(subset=FEATURE_COLS)

print(f"✅ Features engineered for all countries!")
print(f"   Rows before dropna: {len(panel)}")
print(f"   Rows after dropna:  {len(panel_clean)}")
print(f"   Features:           {FEATURE_COLS}")
print(f"   Distress rate:      {panel_clean['distress'].mean():.1%}")
print(f"\nRows per country after cleaning:")
print(panel_clean.groupby('country')['distress']
      .agg(['count','sum'])
      .rename(columns={'count':'rows','sum':'distress_yrs'})
      .to_string())

✅ Features engineered for all countries!
   Rows before dropna: 155
   Rows after dropna:  145
   Features:           ['unemployment_roll3_mean', 'gdp_growth_roll3_std', 'consumption_growth_roll3_mean', 'forex_rapid_decline', 'remittances_pct_gdp_lag1']
   Distress rate:      17.9%

Rows per country after cleaning:
            rows  distress_yrs
country                       
Bangladesh    29             3
India         29             6
Nepal         29             7
Pakistan      29             4
Sri Lanka     29             6


In [8]:
# ================================================
# TRAIN ON SOUTH ASIA, TEST ON NEPAL
# ================================================
# Strategy:
# Training set = all countries EXCEPT Nepal
# Test set     = Nepal only (LOOCV)
#
# This is transfer learning —
# model learns patterns from 4 countries
# then we check if those patterns work for Nepal
# ================================================

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (f1_score, balanced_accuracy_score,
                              recall_score, confusion_matrix)

# Split: non-Nepal for training, Nepal for testing
non_nepal = panel_clean[panel_clean['country'] != 'Nepal']
nepal_only = panel_clean[panel_clean['country'] == 'Nepal']

X_train_panel = non_nepal[FEATURE_COLS]
y_train_panel = non_nepal['distress']

X_test_nepal  = nepal_only[FEATURE_COLS]
y_test_nepal  = nepal_only['distress']

print(f"Training set (non-Nepal): {X_train_panel.shape}")
print(f"  Distress rate: {y_train_panel.mean():.1%}")
print(f"\nTest set (Nepal only): {X_test_nepal.shape}")
print(f"  Distress rate: {y_test_nepal.mean():.1%}")

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_panel)
X_test_scaled  = scaler.transform(X_test_nepal)

# Train XGBoost on South Asia data
xgb_panel = XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.1,
    scale_pos_weight=3,
    random_state=42,
    eval_metric='logloss',
    verbosity=0
)
xgb_panel.fit(X_train_scaled, y_train_panel)

# Predict on Nepal
preds_panel  = xgb_panel.predict(X_test_scaled)
probs_panel  = xgb_panel.predict_proba(X_test_scaled)[:,1]

# Metrics
f1_panel      = f1_score(y_test_nepal, preds_panel, zero_division=0)
bal_panel     = balanced_accuracy_score(y_test_nepal, preds_panel)
recall_panel  = recall_score(y_test_nepal, preds_panel, zero_division=0)
cm_panel      = confusion_matrix(y_test_nepal, preds_panel)

print(f"\n{'='*55}")
print(f"  SOUTH ASIA MODEL — Tested on Nepal")
print(f"{'='*55}")
print(f"  F1 Score:          {f1_panel:.3f}")
print(f"  Balanced Accuracy: {bal_panel:.3f}")
print(f"  Distress Recall:   {recall_panel:.3f}")
print(f"\n  Confusion Matrix:")
print(f"              Predicted")
print(f"              Normal  Distress")
print(f"  Actual Normal    {cm_panel[0][0]:>4}    {cm_panel[0][1]:>4}")
print(f"  Actual Distress  {cm_panel[1][0]:>4}    {cm_panel[1][1]:>4}")

print(f"\n  Misclassified Nepal years:")
nepal_years = nepal_only.index.year.tolist()
for yr, actual, pred in zip(nepal_years,
                             y_test_nepal.tolist(),
                             preds_panel.tolist()):
    if actual != pred:
        kind = "FALSE ALARM" if pred==1 else "MISSED"
        print(f"    {yr}: actual={actual}, predicted={pred}  ← {kind}")

Training set (non-Nepal): (116, 5)
  Distress rate: 16.4%

Test set (Nepal only): (29, 5)
  Distress rate: 24.1%

  SOUTH ASIA MODEL — Tested on Nepal
  F1 Score:          0.250
  Balanced Accuracy: 0.571
  Distress Recall:   0.143

  Confusion Matrix:
              Predicted
              Normal  Distress
  Actual Normal      22       0
  Actual Distress     6       1

  Misclassified Nepal years:
    2015: actual=1, predicted=0  ← MISSED
    2016: actual=1, predicted=0  ← MISSED
    2020: actual=1, predicted=0  ← MISSED
    2021: actual=1, predicted=0  ← MISSED
    2022: actual=1, predicted=0  ← MISSED
    2023: actual=1, predicted=0  ← MISSED


In [9]:
# ================================================
# APPROACH 2 — TRAIN ON ALL 5 COUNTRIES
# TEST ON NEPAL USING LOOCV
# ================================================
# Better approach: include Nepal in training
# but use LOOCV so each Nepal year is tested
# on a model that never saw that specific year
# ================================================

from sklearn.model_selection import LeaveOneOut

# Get Nepal indices in panel
nepal_indices = panel_clean[panel_clean['country'] == 'Nepal'].index

X_panel = panel_clean[FEATURE_COLS]
y_panel = panel_clean['distress']

predictions_all = []
actuals_all     = []
years_all       = []
probs_all       = []

# For each Nepal year, train on everything else
for i, idx in enumerate(nepal_indices):
    # Test = this Nepal year
    test_mask  = panel_clean.index == idx
    train_mask = ~test_mask

    X_train = X_panel[train_mask]
    y_train = y_panel[train_mask]
    X_test  = X_panel[test_mask]
    y_test  = y_panel[test_mask]

    # Scale inside loop
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s  = scaler.transform(X_test)

    # Train and predict
    model = XGBClassifier(
        n_estimators=100, max_depth=3,
        learning_rate=0.1, scale_pos_weight=3,
        random_state=42, eval_metric='logloss',
        verbosity=0
    )
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)[0]
    prob = model.predict_proba(X_test_s)[0][1]

    predictions_all.append(pred)
    actuals_all.append(y_test.values[0])
    years_all.append(idx.year)
    probs_all.append(prob)

# Metrics
f1_all   = f1_score(actuals_all, predictions_all, zero_division=0)
bal_all  = balanced_accuracy_score(actuals_all, predictions_all)
rec_all  = recall_score(actuals_all, predictions_all, zero_division=0)
cm_all   = confusion_matrix(actuals_all, predictions_all)

print(f"{'='*55}")
print(f"  PANEL MODEL (All 5 countries) — Nepal LOOCV")
print(f"{'='*55}")
print(f"  F1 Score:          {f1_all:.3f}")
print(f"  Balanced Accuracy: {bal_all:.3f}")
print(f"  Distress Recall:   {rec_all:.3f}")
print(f"\n  Confusion Matrix:")
print(f"              Predicted")
print(f"              Normal  Distress")
print(f"  Actual Normal    {cm_all[0][0]:>4}    {cm_all[0][1]:>4}")
print(f"  Actual Distress  {cm_all[1][0]:>4}    {cm_all[1][1]:>4}")
print(f"\n  Misclassified Nepal years:")
for yr, actual, pred in zip(years_all, actuals_all, predictions_all):
    if actual != pred:
        kind = "FALSE ALARM" if pred==1 else "MISSED"
        print(f"    {yr}: actual={actual}, predicted={pred}  ← {kind}")

  PANEL MODEL (All 5 countries) — Nepal LOOCV
  F1 Score:          0.500
  Balanced Accuracy: 0.655
  Distress Recall:   0.500

  Confusion Matrix:
              Predicted
              Normal  Distress
  Actual Normal      17       4
  Actual Distress     4       4

  Misclassified Nepal years:
    1998: actual=0, predicted=1  ← FALSE ALARM
    2002: actual=0, predicted=1  ← FALSE ALARM
    2010: actual=1, predicted=0  ← MISSED
    2012: actual=0, predicted=1  ← FALSE ALARM
    2015: actual=1, predicted=0  ← MISSED
    2019: actual=1, predicted=0  ← MISSED
    2022: actual=1, predicted=0  ← MISSED
    2025: actual=0, predicted=1  ← FALSE ALARM


In [10]:
# ================================================
# FINAL COMPARISON — ALL APPROACHES
# ================================================

comparison = pd.DataFrame([
    {"Approach": "Nepal-only XGBoost (LOOCV)",
     "F1": 0.714, "Bal_Acc": 0.814, "Recall": 0.714,
     "Training_rows": 29, "Note": "Best model"},
    {"Approach": "Panel — All 5 Countries (Nepal LOOCV)",
     "F1": 0.500, "Bal_Acc": 0.655, "Recall": 0.500,
     "Training_rows": 145, "Note": "More data but noisier"},
    {"Approach": "South Asia only (no Nepal in training)",
     "F1": 0.250, "Bal_Acc": 0.571, "Recall": 0.143,
     "Training_rows": 116, "Note": "Transfer learning failed"},
])

print("="*65)
print("  PANEL DATA EXPERIMENT — FINAL RESULTS")
print("="*65)
print(comparison.to_string(index=False))
print("="*65)
print("\n🏆 CONCLUSION: Nepal-only model wins")
print("   Adding regional data hurts performance")
print("   Nepal's distress is structurally unique")
print("\n📝 RESEARCH FINDING:")
print("   Nepal-specific features (remittances, forex)")
print("   outperform regional South Asian patterns.")
print("   Country-specific models needed for small")
print("   remittance-dependent economies.")

# Save
comparison.to_csv('data/processed/panel_results.csv', index=False)
print("\n✅ Results saved → data/processed/panel_results.csv")

  PANEL DATA EXPERIMENT — FINAL RESULTS
                              Approach    F1  Bal_Acc  Recall  Training_rows                     Note
            Nepal-only XGBoost (LOOCV) 0.714    0.814   0.714             29               Best model
 Panel — All 5 Countries (Nepal LOOCV) 0.500    0.655   0.500            145    More data but noisier
South Asia only (no Nepal in training) 0.250    0.571   0.143            116 Transfer learning failed

🏆 CONCLUSION: Nepal-only model wins
   Adding regional data hurts performance
   Nepal's distress is structurally unique

📝 RESEARCH FINDING:
   Nepal-specific features (remittances, forex)
   outperform regional South Asian patterns.
   Country-specific models needed for small
   remittance-dependent economies.

✅ Results saved → data/processed/panel_results.csv
